In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from google.colab import files
files.upload()




TypeError: 'NoneType' object is not subscriptable

In [ ]:
import zipfile

with zipfile.ZipFile("ecg_data.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
import pandas as pd

# Load the training CSV file
df = pd.read_csv("mitbih_train.csv", header=None)

# Split features (first 187 columns) and labels (last column)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values


In [ ]:
import matplotlib.pyplot as plt
# viewing first  plots
plt.figure(figsize=(12, 5))
for i in range(5):
    plt.plot(X[i], label=f"Label: {int(y[i])}")
plt.title("Sample ECG Signals from Training Data")
plt.xlabel("Time Steps")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# viewing normal vs abnormal plots

plt.figure(figsize=(12, 5))
for label in range(5):
    index = list(y).index(label)  #  index of first occurrence of this label
    plt.plot(X[index], label=f"Label: {int(y[index])}")
plt.title("Sample ECG Signal of Each Class")
plt.xlabel("Time Steps")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()


Data Preprocess

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Normalize the data (very important for CNNs)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reshape for CNN (CNN needs 3D input: samples, timesteps, channels)
X_scaled = X_scaled.reshape(-1, 187, 1)

# Split into train and test sets (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)


CNN Building

Defining architecture

In [ ]:
model = Sequential()

#  1st Convolutional Block
model.add(Conv1D(filters=32, kernel_size=5, activation='relu', input_shape=(187, 1)))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.2))

# 2nd Convolutional Block
model.add(Conv1D(filters=64, kernel_size=5, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.2))

#  Fully Connected Layers
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(5, activation='softmax'))  # 5 classes


Compiling model

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',  # Because labels are 0,1,2,3,4 (not one-hot)
    optimizer='adam',
    metrics=['accuracy']
)


Training Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test)
)


Evaluating Model

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


Implementin Saliency Map

In [ ]:
# Picking 1 ECG signal from the test set
idx = 0
sample = X_test[idx:idx+1]  # Keeping 3D shape
label = y_test[idx]

print("Actual Label:", label)


In [ ]:
# Creating gradient tape for model attraction


# Converting to tensor and enable gradient tracking
input_tensor = tf.convert_to_tensor(sample)
input_tensor = tf.cast(input_tensor, tf.float32)

# Useig GradientTape to compute gradient of output w.r.t. input
with tf.GradientTape() as tape:
    tape.watch(input_tensor)
    predictions = model(input_tensor)
    class_index = tf.argmax(predictions[0])
    class_output = predictions[:, class_index]

# Get the gradient (important parts of input)
grads = tape.gradient(class_output, input_tensor)[0]

# Take absolute value and normalize
saliency = np.abs(grads.numpy())
saliency = saliency / np.max(saliency)


Plotting

In [ ]:
# Plot ECG + amplified saliency map
plt.figure(figsize=(14, 5))
plt.plot(sample[0], label='ECG Signal', linewidth=1.5)

# Amplify saliency map for visibility
plt.plot(saliency[0] * np.max(sample[0]), label='Saliency Map', color='red', alpha=0.7, linewidth=2)

plt.title("ECG Signal with Saliency Map (Model's Focus)")
plt.xlabel("Time Steps")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()



Improving vizuals

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

#  'sample' is  input ECG of shape (1, 187, 1)
# and 'model' is our trained CNN

# Getting the model prediction for the sample
sample = X_test[0].reshape(1, 187, 1)  #  any sample from test set

#  gradient (saliency map) of the output class w.r.t. input
input_tensor = tf.convert_to_tensor(sample)
with tf.GradientTape() as tape:
    tape.watch(input_tensor)
    predictions = model(input_tensor)
    class_idx = tf.argmax(predictions[0])
    loss = predictions[:, class_idx]

grads = tape.gradient(loss, input_tensor)
saliency = tf.abs(grads).numpy()

#  Normalizing safely
min_val = np.min(saliency[0])
max_val = np.max(saliency[0])
if max_val - min_val == 0:
    norm_saliency = np.zeros_like(saliency[0])
else:
    norm_saliency = (saliency[0] - min_val) / (max_val - min_val)

#  Overlay heat zone plot
plt.figure(figsize=(14, 5))
plt.plot(sample[0], label='ECG Signal', color='blue')

plt.fill_between(
    range(187),
    sample[0].flatten(),
    color='red',
    alpha=norm_saliency.flatten(),  # must flatten for plotting
    label='Saliency Zone'
)

plt.title("ECG Signal with Heatmap Overlay (Model's Focus)")
plt.xlabel("Time Steps")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()


NameError: name 'X_test' is not defined

Saving

In [ ]:
model.save("ecg_cnn_model.keras")


In [ ]:
from google.colab import files
files.download("ecg_cnn_model.keras")
